# Preprocessing

Loads raw data from `DATA_ROOT` and prepares the analysis-ready tables.

`DATA_ROOT` is set in two ways depending on environment:
- **Local**: read from `.env` in the project root (copy `.env.example` → `.env` and set your path)
- **Colab**: mount Google Drive in the cell below, then set `DATA_ROOT` to the Drive path

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/My Drive/my_projects/M.A. Parliament/Code and Data/data"
else:
    if "DATA_ROOT" not in os.environ:
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv())
        except ImportError:
            pass
    DATA_ROOT = os.environ.get("DATA_ROOT", "")

print("DATA_ROOT:", DATA_ROOT)
assert DATA_ROOT and os.path.isdir(DATA_ROOT), f"DATA_ROOT not set or missing: {DATA_ROOT!r}"

DATA_ROOT: /Users/anna/Library/CloudStorage/GoogleDrive-anle.werner.01@gmail.com/My Drive/my_projects/M.A. Parliament/Code and Data/data


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW  = Path(DATA_ROOT) / 'raw'
PROC = Path(DATA_ROOT) / 'processed'
V3   = RAW / 'stateparl_v3_parquet'   # StateParl v3 (current)

## StateParl v3 — paragraphs

16,078,467 rows × 14 columns (2000–2025, all 16 states).

Key v3 column changes from v2:
- `id` → `paragraph_id`, `protocol` → `protocol_id`, `sequence_number` → `protocol_position`
- `speaker_name` → `speaker_paragraph`, `speaker_id` → `mandate_id`
- New: `speech_id` (FK to speeches table), `segment_position`, `page`
- Affiliation `ijn` renamed to `nsc` (non-speech content)

In [4]:
paragraphs = pd.read_parquet(V3 / "stateparl_v3_paragraphs.parquet")
print(paragraphs.shape)
print(paragraphs.columns.tolist())
paragraphs.head(15)

(16078467, 14)
['paragraph_id', 'protocol_id', 'state', 'period', 'nth', 'date', 'speech_id', 'protocol_position', 'segment_position', 'page', 'speaker_paragraph', 'mandate_id', 'affiliation', 'content']


,paragraph_id,protocol_id,state,period,nth,date,speech_id,protocol_position,segment_position,page,speaker_paragraph,mandate_id,affiliation,content
0,1,bb_3_10,bb,3,10,2000-02-24,NaN,1,2,4,Knoblich,bb_3_pre_knoblich,pre,Werte Kolleginnen und Kollegen! Ich begrüße Si...
1,2,bb_3_10,bb,3,10,2000-02-24,NaN,2,3,4,,,nsc,(Allgemeiner Beifall)
2,3,bb_3_10,bb,3,10,2000-02-24,NaN,3,4,4,Knoblich,bb_3_pre_knoblich,pre,Nicht weniger herzlich begrüße ich unsere Stam...
3,4,bb_3_10,bb,3,10,2000-02-24,NaN,4,5,4,Knoblich,bb_3_pre_knoblich,pre,Mit der Einladung ist Ihnen der Vorschlag zur ...
4,5,bb_3_10,bb,3,10,2000-02-24,NaN,5,6,4,Knoblich,bb_3_pre_knoblich,pre,"Ich darf darauf hinweisen, dass die DVU-Frakti..."
5,6,bb_3_10,bb,3,10,2000-02-24,NaN,6,7,4,Knoblich,bb_3_pre_knoblich,pre,Damit darf ich um Ihr zustimmendes Handzeichen...
6,7,bb_3_10,bb,3,10,2000-02-24,NaN,7,8,4,Knoblich,bb_3_pre_knoblich,pre,"Ich merke gerade, dass es notwendig gewesen wä..."
7,8,bb_3_10,bb,3,10,2000-02-24,NaN,8,9,4,Knoblich,bb_3_pre_knoblich,pre,"Es wird vorgeschlagen, einen neuen Tagesordnun..."
8,9,bb_3_10,bb,3,10,2000-02-24,NaN,9,10,4,Knoblich,bb_3_pre_knoblich,pre,Außerdem wird ein neuer Tagesordnungspunkt 11 ...
9,10,bb_3_10,bb,3,10,2000-02-24,NaN,10,11,4,Knoblich,bb_3_pre_knoblich,pre,"Schließlich wird vorgeschlagen, einen Tagesord..."


## Annotator datasets (2010 / 2018 / 2021)

Full calendar year, all 16 states, sourced from v3. Feeds the annotator's
`2010` / `2018` / `2021` dataset options. Replaces the old v2-based
`annotations_input.csv` (2021) and `annotations_input_2018.csv` (2018, which
also carried model labels — kept around, not reused here) and adds a new
pre-AfD-entry `2010` reference year.

In [ ]:
bb_2018 = bb[bb['date'] == '2018-06-27'].copy()
print('BB 2018 rows:', len(bb_2018))

out_path = Path(DATA_ROOT) / 'labelling' / 'annotations_input_2018_bb.csv'
bb_2018.to_csv(out_path, index=False)
print('Saved to', out_path)

dates = pd.to_datetime(paragraphs['date'])

for year in (2010, 2018, 2021):
    subset = paragraphs[dates.dt.year == year].copy()
    print(f'{year}: {len(subset)} rows, {subset["state"].nunique()} states')
    out_path = Path(DATA_ROOT) / 'labelling' / f'annotations_input_{year}_v3.csv'
    subset.to_csv(out_path, index=False)
    print('Saved to', out_path)

In [ ]:
protocols = pd.read_parquet(V3 / 'stateparl_v3_protocols.parquet')
bb_protocols = protocols[protocols['state'] == 'bb'].sort_values(['period', 'nth'])
bb_protocols[['protocol_id', 'period', 'nth', 'date']].head()

## StateParl v3 — speeches

1,072,934 rows. Groups all consecutive paragraphs by the same speaker into one addressable
unit per speaker turn. `speech_id` in the paragraphs table links every paragraph (including
`nsc` interruptions) back to the speech it belongs to or interrupts.

In [ ]:
speeches = pd.read_parquet(V3 / 'stateparl_v3_speeches.parquet')
print(speeches.shape)
speeches.head(2)

## StateParl v3 — mandates

17,543 rows. Replaces v2's `mandateMappings`. Every distinct speaker mandate with
`mandate_id` key (`{state}_{period}_{affiliation}_{name}`), canonical affiliation,
and optional `statepol_id` link to the StatePol politician database.

In [ ]:
mandates = pd.read_parquet(V3 / 'stateparl_v3_mandates.parquet')
print(mandates.shape)
mandates.head(2)

## Non-speech content (interjections)

v3 affiliation code `nsc` (was `ijn` in v2). Still raw stenographic text — parsing and
structured extraction is done by `measurement/preprocessing.py`. Each `nsc` row carries
`speech_id`, linking directly to the speech being interrupted — no join needed.

In [ ]:
interjections_df = paragraphs[paragraphs["affiliation"] == "nsc"].copy()
print(interjections_df.shape)
interjections_df[['paragraph_id', 'speech_id', 'state', 'period', 'nth',
                   'protocol_position', 'affiliation', 'content']].head()

## Brandenburg motion metadata (scraped)

In [ ]:
bb_motions = pd.read_csv(RAW / 'bb_motions_metadata.csv', low_memory=False)
print(bb_motions.shape)
# wahlperiode, vorgang_id, vtyp, dok_nr, dok_typ, datum, titel,
# urheber, keywords, fundstelle, pdf_url, all_pdf_urls
bb_motions.head(2)

## Bavaria motion metadata (scraped)

In [ ]:
by_motions = pd.read_csv(RAW / 'by_motions_metadata.csv', low_memory=False)
print(by_motions.shape)
by_motions.head(2)